In [1]:
import os
import pandas as pd
from dotenv import load_dotenv

# 1. Initialize and Test Environment Variables
# We use dotenv_path to explicitly point up one directory to the root .env file
env_path = os.path.join("..", ".env")
load_dotenv(dotenv_path=env_path)

print("--- Environment Check ---")
# Safely check if the variables loaded (without printing the actual keys)
etherscan_status = "Loaded" if os.getenv("ETHERSCAN_API_KEY") else "Missing"
print(f"Etherscan API Key Status: {etherscan_status}")
print("-" * 50)

# 2. Define the exact path to raw data
data_dir = os.path.join("..", "data", "01_raw")

print(f"\n--- Initiating Rigorous Diagnostic Scan of: {data_dir} ---\n")

if not os.path.exists(data_dir):
    print("CRITICAL ERROR: Path not found. Check your relative directories.")
else:
    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                display_name = os.path.join(os.path.basename(root), file)
                print(f"Analyzing: {display_name}")
                
                try:
                    df = pd.read_csv(file_path)
                    print(f"  * Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
                    print(f"  * Columns: {list(df.columns)}")
                    
                    null_count = df.isna().sum().sum()
                    if null_count > 0:
                        print(f"  * WARNING: {null_count} missing values detected.")
                    else:
                        print(f"  * Data Integrity: 0 missing values.")
                        
                    print("-" * 50)
                except Exception as e:
                    print(f"  * ERROR reading file: {e}")
                    print("-" * 50)
                    
print("\n--- Diagnostic Scan Complete ---")

--- Environment Check ---
Etherscan API Key Status: Loaded
--------------------------------------------------

--- Initiating Rigorous Diagnostic Scan of: ..\data\01_raw ---

Analyzing: coinmetrics\active_addresses.csv
  * Dimensions: 8768 rows, 3 columns
  * Columns: ['asset', 'time', 'AdrActCnt']
  * Data Integrity: 0 missing values.
--------------------------------------------------
Analyzing: coinmetrics\eth_fees.csv
  * Dimensions: 2192 rows, 4 columns
  * Columns: ['asset', 'time', 'FeeTotNtv', 'TxCnt']
  * Data Integrity: 0 missing values.
--------------------------------------------------
Analyzing: coinmetrics\eth_trx_price_usd.csv
  * Dimensions: 4384 rows, 3 columns
  * Columns: ['asset', 'time', 'PriceUSD']
  * Data Integrity: 0 missing values.
--------------------------------------------------
Analyzing: coinmetrics\supply.csv
  * Dimensions: 8768 rows, 3 columns
  * Columns: ['asset', 'time', 'SplyCur']
  * Data Integrity: 0 missing values.
-------------------------------

In [3]:
import os
import pandas as pd
import yfinance as yf

print("--- Initiating Keyless Volume Extraction (Yahoo Finance) ---\n")

# Set up the directory (we will use a yfinance folder instead of coingecko)
base_dir = os.path.join("..", "data", "01_raw")
yf_dir = os.path.join(base_dir, "yfinance")
os.makedirs(yf_dir, exist_ok=True)

# Define our stablecoin tickers on Yahoo Finance
tickers = {
    "USDT": "USDT-USD",
    "USDC": "USDC-USD"
}

for symbol, ticker in tickers.items():
    try:
        print(f"Fetching {symbol} daily volume...")
        # Download data for our exact structural break timeframe
        data = yf.download(ticker, start="2020-01-01", end="2025-12-31", progress=False)
        
        # Isolate the Volume column and clean the dataframe
        df_vol = data[['Volume']].reset_index()
        # Handle yfinance's multi-level column indexing if present
        if isinstance(df_vol.columns, pd.MultiIndex):
            df_vol.columns = df_vol.columns.get_level_values(0)
            
        df_vol.columns = ['timestamp', 'volume_usd']
        df_vol['symbol'] = symbol
        
        # Save to our raw data folder
        out_path = os.path.join(yf_dir, f"{symbol}_daily_volume.csv")
        df_vol.to_csv(out_path, index=False)
        print(f"  [SUCCESS] Saved {symbol} volume data ({len(df_vol)} rows)")
        
    except Exception as e:
        print(f"  [ERROR] Failed to fetch {symbol}: {e}")

print("\n--- Phase 1.2 Officially Complete ---")

--- Initiating Keyless Volume Extraction (Yahoo Finance) ---

Fetching USDT daily volume...
  [SUCCESS] Saved USDT volume data (2191 rows)
Fetching USDC daily volume...
  [SUCCESS] Saved USDC volume data (2191 rows)

--- Phase 1.2 Officially Complete ---
